# Experiments runner

Batch runner for the numerical experiments. The scripts are divided into four main parts according to the solver being used: 
- MUMPS
- SUPERLU_DIST
- PASTIX
- MKL PARDISO

Solver-specific options are set for all of these scripts.

Set `-own_reordering true` to apply reordering via the custom `reorder()` routine before solving.  
Set `-own_reordering false` (default) to let the solver handle reordering internally.

---

## Configuration

In [19]:
import os
import subprocess

In [20]:
# data paths
MAT_INPUT_PATH = "../matrices/bin/"
VEC_INPUT_PATH = "../matrices/bin/vec/"
MAT_PERM_PATH  = "../matrices/bin/permuted/"

#PETSC_DIR  = "/home/alexandr/Documents/bsc_thesis/numerical_experiments/petsc"
PETSC_DIR  = "../petsc"
PETSC_ARCH = "arch-opt" 

MPIEXEC = f"{PETSC_DIR}/{PETSC_ARCH}/bin/mpiexec"
EXE_PATH = "../src/experiment"
EXE = f"{MPIEXEC} -n {{nproc}} {EXE_PATH}"

## Helper function

In [21]:
def run_experiments(
        exe, 
        solver, 
        nproc, 
        solver_options, 
        log_dir, 
        ordering_tag, 
        pc_type="lu",
        own_reordering=False, 
        ordering_type=None, 
        rhs_file=None):
    """
    Run the experiment executable over all .bin matrices in MAT_INPUT_PATH.

    Parameters
    ----------
    exe             : str   Executable template.
    solver          : str   PETSc mat_solver_type (e.g. 'mumps').
    nproc           : int   Number of MPI processes.
    solver_options  : str   Solver-specific flags.
    log_dir         : str   Directory where log files are saved.
    ordering_tag    : str   Short label used in the log filename.
    pc_type         : str   Preconditioner type (LU by default).
    own_reordering  : bool  If True, use custom reorder() before solving.
    ordering_type   : str   PETSc ordering type (only used if own_reordering=True).
    rhs_file        : str   Optional path to a .bin RHS vector.
    """
    os.makedirs(log_dir, exist_ok=True)

    mat_files = sorted(f for f in os.listdir(MAT_INPUT_PATH) if f.endswith(".bin"))

    for mat_file in mat_files:
        mat_name = os.path.splitext(mat_file)[0]
        mat_path = os.path.join(MAT_INPUT_PATH, mat_file)
        log_file = f"{ordering_tag}_{mat_name}_{nproc}procs.log"
        log_path = os.path.join(log_dir, log_file)

        cmd_parts = [
            exe.format(nproc=nproc),
            f"-mat_file {mat_path}",
            f"-pc_type {pc_type}",
            f"-mat_solver_type {solver}",
            "-get_solution_norm true",
        ]

        if rhs_file:
            cmd_parts.append(f"-rhs_file {rhs_file}")

        if own_reordering and ordering_type:
            cmd_parts += [
                "-own_reordering true",
                f"-mat_ordering_type {ordering_type}",
            ]

        if solver_options:
            cmd_parts.append(solver_options)

        cmd = " ".join(cmd_parts)

        print(f"Running {mat_file} -> {log_file}")

        env = os.environ.copy()

        with open(log_path, "w") as f:
            subprocess.run(cmd, shell=True, stdout=f, stderr=subprocess.STDOUT, env=env)

    print("======= COMPLETE =======")

## MUMPS runs

In [22]:
MUMPS_OPTIONS = (
    "-mat_mumps_icntl_4 2 "     # verbosity
    "-mat_mumps_icntl_6 2 "     # permute to zero-free diagonal and scale the matrix
    "-mat_mumps_icntl_7 0 "     # compute a symmetric permutation in sequential analysis
                                # 0=AMD, 2=AMF, 3=Scotch, 4=PORD, 5=Metis, 6=QAMD, and 7=auto
                                # use -pc_factor_mat_ordering_type to have PETSc perform the ordering (sequential only)
    "-mat_mumps_icntl_11 "      # statistics related to an error analysis (via -ksp_view)
    "-mat_mumps_icntl_22 1 "    # in-core/out-of-core factorization and solve (0 or 1)
    "-mat_mumps_icntl_28 1 "    # 1 = sequential analysis and ICNTL(7) ordering
                                # 2 = parallel analysis and ICNTL(29) ordering
    "-mat_mumps_icntl_29 2 "    # parallel ordering 1 = ptscotch, 2 = parmetis

)

run_experiments(
    exe           = EXE,
    solver        = "mumps",
    nproc         = 4,
    pc_type       = "lu",
    solver_options= MUMPS_OPTIONS,
    log_dir       = "./mumps",
    ordering_tag  = "amd",
)

Running msdoor.bin -> amd_msdoor_4procs.log
======= COMPLETE =======


## SUPERLU_DIST runs

In [23]:
SUPERLU_OPTIONS = (
    "-mat_superlu_dist_equil true "             # equilibrate the matrix
    "-mat_superlu_dist_iterrefine true "
    "-mat_superlu_dist_replacetinypivot "
    "-mat_superlu_dist_rowperm LargeDiag_MC64 " # row permutation
                                                # NOROWPERM, LargeDiag_MC64, LargeDiag_AWPM,MY_PERMR
    "-mat_superlu_dist_colperm MMD_AT_PLUS_A "       # column permutation
                                                # NATURAL, MMD_AT_PLUS_A, MMD_ATA,METIS_AT_PLUS_A, PARMETIS
    "-mat_superlu_dist_iterrefine "             # use iterative refinement
    "-mat_superlu_dist_printstat "           # print factorization information
)

run_experiments(
    exe           = EXE,
    solver        = "superlu_dist",
    nproc         = 4,
    pc_type       = "lu",
    solver_options= SUPERLU_OPTIONS,
    log_dir       = "./superlu_dist",
    ordering_tag  = "LargeDiag_MC64_MMD_AT_PLUS_A",
)

Running msdoor.bin -> LargeDiag_MC64_MMD_AT_PLUS_A_msdoor_4procs.log
======= COMPLETE =======


## PaStiX runs

In [24]:
PASTIX_OPTIONS = (
    "-mat_pastix_verbose 2 "        # verbosity (1, 2, 3)
    "-mat_pastix_factorization 2 "  # factorization
                                    # 0 - Cholesky, 1 - LDL^t, 2 - LU, 3 - LL^t, 4 - LDL^h
    "-mat_pastix_ordering 1 "       # 0 - Scotch, 1 - Metis
    "-mat_pastix_itermax "          # max number of iterations during refinement
    "-mat_pastix_thread_nbr 1"      # 1 thread per MPI process
)

run_experiments(
    exe           = EXE,
    solver        = "pastix",
    nproc         = 8,
    solver_options= PASTIX_OPTIONS,
    log_dir       = "./pastix",
    ordering_tag  = "metis",
)

Running msdoor.bin -> metis_msdoor_8procs.log
======= COMPLETE =======


## MKL PARDISO

TBA

PETSc solver type:
- `mkl_cpardiso` — MPI (Cluster PARDISO)
- `mkl_pardiso`  — sequential

In [25]:
use_external_ordering = False
ordering_type         = "nd"
mat_solver            = "mkl_cpardiso"

PARDISO_OPTIONS = (
    "-mat_mkl_pardiso_68 1 "
)

ordering_tag = f"external_{ordering_type}" if use_external_ordering else "internal"

run_experiments(
    exe             = EXE,
    solver          = mat_solver,
    nproc           = 4
    solver_options  = PARDISO_OPTIONS,
    log_dir         = "./pardiso",
    ordering_tag    = ordering_tag,
    own_reordering  = use_external_ordering,
    ordering_type   = ordering_type if use_external_ordering else None,
)

SyntaxError: invalid syntax. Perhaps you forgot a comma? (3205874824.py, line 14)